# 🏥 TEST JURY - Modèle Médical TechCorp (Hackathon 2026)

Ce notebook permet d'évaluer le modèle expérimental médical **Phi-3.5-Medical-TechCorp** directement dans le cloud (Windows, Mac, Linux), sans avoir besoin de carte graphique locale.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes

## 📥 Téléchargement des poids de votre dépôt

In [ ]:
# Récupération de vos adaptateurs LoRA entraînés avec MLX
!rm -rf hackathon_ynov
!git clone https://github.com/VOTRE_PSEUDO/VOTRE_REPO.git hackathon_ynov
# Note pour le jury : Si le repo est privé, vous pouvez uploader manuellement le dossier 'adapters/' ici.

## 🚀 Chargement de l'IA (Base + LoRA Médical)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_id = "microsoft/Phi-3.5-mini-instruct"
adapter_dir = "hackathon_ynov/adapters"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Chargement du modèle de base (4-bit)...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Injection du savoir médical (LoRA)...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

print("✅ Modèle prêt pour l'évaluation clinique !")

## 🩺 Lancer le test interactif

In [ ]:
def test_model(question):
    prompt = f"<|user|>\n{question}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=250, 
        temperature=0.3
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extraire uniquement la réponse de l'assistant
    print("\n" + "="*50)
    print("🤖 DIAGNOSTIC IA :")
    print(response.split("<|assistant|>")[-1].strip())
    print("="*50 + "\n")

test_model("Quels sont les symptômes d'une grippe sévère ?")